# Извлечение семантики из параметров запроса и айтема

`item_infm_params_text` и его query-аналог `search_infm_params_text` - это ravel-склейка `label value label value...` без разделителей: поля формы объявления (разные для разных подкатегорий), и, соответственно, выбранные пользователем фильтры поиска.\
 Дамп для эмбеддинга использовал только `item_title_raw`+`item_description_raw` и `search_query` - параметров не было, хотя там есть конкретные названия услуг (чисто взглядом на паркетик)

In [2]:
import sys

sys.path.append("..")

import re
from collections import Counter

import pandas as pd

train = pd.read_parquet("../data/train.parquet")
items = train.drop_duplicates("item_id")[["item_id", "item_title_raw", "item_infm_params_text", "item_category_id", "item_microcat_id"]]
print("train:", len(train), "строк,", len(items), "уникальных items")

train: 497673 строк, 344825 уникальных items


## Как искал лейблы

Текст не структурирован, поэтому единственный способ понять состав - частотный анализ.\
 Идея: лейблы формы - это короткие n-граммы, которые с заглавной буквы и повторяются в **куче** объявлениях (это фиксированный, но большой словарь полей), в отличие от конкретных значений, которые зависят от подкатегории.

In [4]:
print('уникальных item_category_id:', items['item_category_id'].nunique())
print()
print(items['item_category_id'].value_counts().head())
print()
print('уникальных item_microcat_id:', items['item_microcat_id'].nunique())

уникальных item_category_id: 8

item_category_id
114    344814
40          2
27          2
19          2
112         2
Name: count, dtype: int64

уникальных item_microcat_id: 212


`item_category_id` - как мы уже знаем услуги.\
А вот `item_microcat_id` - 212 значений, и именно микрокатегория определяет, какие доп.поля формы показываются продавцу (автоуслуги, фото/видео-съемка).\
Значит, если сэмплировать случайно, редкие микрокатегории просто не попадут в выборку и их специфичные лейблы будут не видны - нужна **stratify_by=microcat_id**.

In [5]:
sample = items.groupby("item_microcat_id", group_keys=False).apply(
    lambda g: g.sample(min(len(g), 60), random_state=1)
)
print("размер стратифицированной выборки:", len(sample))
texts = sample["item_infm_params_text"].tolist()

doc_freq = Counter()
for text in texts:
    words = text.split()
    seen = set()
    for n in (1, 2, 3, 4, 5):
        for i in range(len(words) - n + 1):
            if words[i][0].isupper():
                seen.add(" ".join(words[i:i + n]))
    doc_freq.update(seen)

common = sorted([(p, c) for p, c in doc_freq.items() if c >= 15], key=lambda x: -x[1])
print(f"кандидатов в лейблы (document frequency >= 15 из {len(texts)}): {len(common)}")

размер стратифицированной выборки: 11558
кандидатов в лейблы (document frequency >= 15 из 11558): 17661


Топ по document frequency - это универсальные для всех микрокатегорий поля (`Вид услуги`, `Место оказания услуг`, `Опыт работы`, ...). \
Специфичные же лейблы (авто/фото/дизайн) прячутся дальше по списку, с более низкой общей частотой - их видно только благодаря стратификации:

In [7]:
for p, c in common[:20]:
    print(f"{c:5d} ({c / len(texts) * 100:5.1f}%)  {p}")
print()
for p, c in common[300:315]:
    print(f"{c:5d} ({c / len(texts) * 100:5.1f}%)  {p}")

11556 (100.0%)  Вид
11554 (100.0%)  Место
11547 ( 99.9%)  Вид услуги
11543 ( 99.9%)  Место оказания
11543 ( 99.9%)  Место оказания услуг
11514 ( 99.6%)  Тип
10765 ( 93.1%)  Тип стоимости за
10765 ( 93.1%)  Тип стоимости
 9588 ( 83.0%)  Начальная цена
 9588 ( 83.0%)  Начальная
 9344 ( 80.8%)  Тип стоимости за услугу
 9197 ( 79.6%)  Тип услуги
 8521 ( 73.7%)  Опыт работы
 8521 ( 73.7%)  Опыт
 7390 ( 63.9%)  Стоимость
 7377 ( 63.8%)  Начальная цена Тип
 7323 ( 63.4%)  Начальная цена Тип стоимости
 7315 ( 63.3%)  Начальная цена Тип стоимости за
 6115 ( 52.9%)  Тип стоимости за услугу Стоимость
 5922 ( 51.2%)  Тип стоимости за услугу Начальная

  837 (  7.2%)  Суббота Рабочие дни Воскресенье
  837 (  7.2%)  Суббота Рабочие
  837 (  7.2%)  Суббота Рабочие дни
  832 (  7.2%)  Год
  829 (  7.2%)  Работа с
  829 (  7.2%)  Пятница Рабочие дни Суббота Рабочие
  820 (  7.1%)  Бригада 2–4 человека Время для
  820 (  7.1%)  Бригада 2–4 человека Время
  817 (  7.1%)  Как вы работаете В
  817 (  7.1%)

## Флаг или поле со значением?

Не все лейблы одинаковы:
- чекбоксы, которые **присутствуют только когда отмечены**
- поля с реальным значением после них. Перепутать их важно не меньше, чем найти сам лейбл: если считать чекбокс полем со значением, можно случайно вставить в текст следующий (не относящийся) лейбл как будто это значение. \
Проверяем конкретно `Берёте ли срочные заказы`:

In [13]:
for t in items["item_infm_params_text"]:
    idx = t.find("Берёте ли срочные заказы")
    if idx != -1:
        print(repr(t[idx:idx + 100]))
        break

'Берёте ли срочные заказы Гарантия на выполнение работ Как вы работаете У себя Как вы работаете У кли'


После лейбла сразу идёт **следующий лейбл** (`Предоплата`, `Гарантия на выполнение работ`...), а не 0/1 - значит, это чекбокс, и в текст должно уходить само название лейбла как токен, а не что-то после него.

## Итоговая классификация

После анализа, разделил лейблы на 3 группы:

- **TEXT_LABELS** - значение уходит в текст как есть (`Название услуги`, `Специальность` и др) - это уникальные семантически ценные значения.
- **FLAG_LABELS** - присутствие лейбла добавляется как токен (`Гарантия`, `Предоплата`, и др) - подтверждённые выше чекбоксы без явного значения.
- **NOISE_LABELS** - отбрасывается целиком (числа, расписание, и др.) - чтобы не давать шуму склеиваться с соседним ценным значением.

Покрытие не 100% - редкие подкатегории со своими полями, не попавшими в этот словарь, могут давать склеивать текст у части объявлений. Не хочется рассматривать все кейсы :(

In [21]:
from preprocessing import clean_text, extract_semantic_params

print("примеры raw => extracted\n")
for _, row in items.sample(5, random_state=67).iterrows():
    extracted = extract_semantic_params(row["item_infm_params_text"])
    raw_len = len(row["item_infm_params_text"].split())
    print("TITLE:", row["item_title_raw"])
    print(f"RAW: {raw_len} слов => EXTRACTED: {len(extracted.split())} слов")
    print("EXTRACTED:", extracted)
    print()

примеры raw => extracted

TITLE: Регистрация переоборудования автомобиля за 2 дня
RAW: 35 слов => EXTRACTED: 12 слов
EXTRACTED: Деловые услуги Консультирование Берёте ли срочные заказы Удалённо Предоплата Работа по договору

TITLE: Юрист. Бесплатная консультация
RAW: 45 слов => EXTRACTED: 17 слов
EXTRACTED: Деловые услуги Юридические услуги Берёте ли срочные заказы У себя У клиента Удалённо Предоплата Работа по договору

TITLE: Вспашка земли минитрактором
RAW: 195 слов => EXTRACTED: 23 слов
EXTRACTED: Сад, благоустройство Земляные работы Работа по договору Гарантия Готов закупить материалы Вспашка земли Валка деревьев с корнем Разработка грунта экскаватором По всему городу

TITLE: Кератин
RAW: 103 слов => EXTRACTED: 25 слов
EXTRACTED: Красота, здоровье Услуги парикмахера У заказчика дома У себя дома Женщины Женщина или сфера Другие процедуры для волос Кератиновое выпрямление «Ботокс» для волос Не выезжаю

TITLE: Копка траншей
RAW: 206 слов => EXTRACTED: 23 слов
EXTRACTED: Сад, благоуст

In [24]:
eval_sample = items.sample(5000, random_state=52)
raw_lens = eval_sample["item_infm_params_text"].str.split().str.len()
extracted_series = eval_sample["item_infm_params_text"].apply(extract_semantic_params)
extracted_lens = extracted_series.str.split().str.len().fillna(0)

print("медиана слов RAW:", raw_lens.median())
print("медиана слов EXTRACTED:", extracted_lens.median())
print("доля items с пустым EXTRACTED (после выкидывания шума):", (extracted_lens == 0).mean())
print("доля items с длинным (>40 слов) EXTRACTED (те самые редкие уникальыне лейблы):", (extracted_lens > 50).mean())

медиана слов RAW: 124.0
медиана слов EXTRACTED: 21.0
доля items с пустым EXTRACTED (после выкидывания шума): 0.0
доля items с длинным (>40 слов) EXTRACTED (те самые редкие уникальыне лейблы): 0.1212


## То же самое на стороне запроса

`search_infm_params_text` - та же специфика лейблов (это выбранные фильтры поиска), поэтому `extract_semantic_params()` переиспользуется без изменений.

In [27]:
queries_sample = train[train["search_infm_params_text"].str.strip() != ""].sample(5, random_state=37)
for _, row in queries_sample.iterrows():
    extracted = extract_semantic_params(row["search_infm_params_text"])
    print("QUERY:", row["search_query"])
    print("RAW:", row["search_infm_params_text"])
    print("EXTRACTED:", extracted)
    print()

QUERY: игра на гитаре
RAW: Вид услуги Обучение, курсы
EXTRACTED: Обучение, курсы

QUERY: репетитор итальянского языка
RAW: Предмет или специальность Итальянский язык Тип услуги Иностранные языки Вид услуги Обучение, курсы
EXTRACTED: Иностранные языки Обучение, курсы

QUERY: химчистка мебели
RAW: Тип услуги Чистка мягкой мебели Вид услуги Уборка
EXTRACTED: Чистка мягкой мебели Уборка

QUERY: укладка плитки
RAW: Тип услуги Плиточные работы Вид услуги Ремонт и отделка
EXTRACTED: Плиточные работы Ремонт и отделка

QUERY: восковая эпиляция
RAW: Онлайн-запись Поиск по слотам Завтра Тип услуги Эпиляция Вид услуги Красота, здоровье
EXTRACTED: Эпиляция Красота, здоровье



## Финальный текст для эмбеддинга

**Item**: `title + ". " + description + ". " + extracted_params` (потом `clean_text()` + префикс `passage`).\
**Query**: `search_query + " " + extracted_params` (потом `clean_text()` + префикс `query`).

In [30]:
def build_item_text(row):
    parts = [row["item_title_raw"], ". ", row["item_description_raw"]]
    extracted = extract_semantic_params(row["item_infm_params_text"])
    if extracted:
        parts.append(". " + extracted)
    return clean_text("".join(parts))


def build_query_text(row):
    parts = [row["search_query"]]
    extracted = extract_semantic_params(row["search_infm_params_text"])
    if extracted:
        parts.append(" " + extracted)
    return clean_text("".join(parts))


preview = train.sample(3, random_state=888)
for _, row in preview.iterrows():
    print("ITEM TEXT:", build_item_text(row)[:300])
    print("QUERY TEXT:", build_query_text(row))
    print()

ITEM TEXT: Графический дизайнер/дизайн/баннер/сертификат. Графический дизайнер/дизайн/баннер/сертификат

✨ Меня зовут Дарина, я графический и веб-дизайнер с 4-летним опытом. Создаю дизайн, который работает: усиливает доверие, повышает узнаваемость и конвертирует. 483+ реализованных проекта.

❗️Вы получаете не 
QUERY TEXT: дизайнер Деловые услуги

ITEM TEXT: Ручной рерайт. Повышение оригинальности. Профессиональный рерайт научных работ | Антиплагиат 85%+ | Без следов AI

ChatGPT версии 3.5 высвечивается как сгенерированный текст! Технический рерайт не работает точно! Будьте аккуратны!

✅ Помощь с повышением уникальности текста по системам Antiplagiat, А
QUERY TEXT: убрать ии в тексте

ITEM TEXT: Дипломные и курсовые работы. Поможем с оформлением любых учебных работ!

⏰ Срочные заказы – от 1 дня! Гарантируем своевременную сдачу!

В каких видах работ мы оказываем профессиональную помощь:

• Оформление курсовых и дипломных работ любой сложности
• Подготовка выпускных квалификационных работ

## Дальше

Это только preproc корпуса, дальше опять закидываю на kaggle :)\
Эх, почему у меня нет своей a100-80gb :(